/Users/hyin/usgs_mendenhall/events/2026-04-14_nevada/wisp_inversion/mask_s2_observations.py

In [1]:
import geopandas as gpd
from shapely.geometry import Point

insar_data = "/Volumes/T9_InSAR/2026-04-14_nevada/gcent/products/nisar-p042-20260406-20260418_resampled.txt"
insar_out = "/Volumes/T9_InSAR/2026-04-14_nevada/gcent/products/nisar-p042-20260406-20260418_resampled_crop.txt"

# mask_file = "/Users/hyin/usgs_mendenhall/events/2026_venezuela/ffm/resampled_interferograms/simple_fault.shp"       # line or polygon shapefile
mask_file = None
buffer_distance = 5000        # meters

crop = True
bl = [39.20715,-119.22306]  # lat, lon (copied from QGIS)
tr = [39.44069,-118.87245]

# ---------------------------------------------------------
# Read mask geometry
# ---------------------------------------------------------
if mask_file is not None:
    mask = gpd.read_file(mask_file)

    # Project to a metric CRS
    mask = mask.to_crs("EPSG:32646")

    # Merge all features into one geometry
    geometry = mask.union_all()

    # Buffer only if it is a line
    if geometry.geom_type in ["LineString", "MultiLineString"]:
        geometry = geometry.buffer(buffer_distance)

# ---------------------------------------------------------
# Apply bounding box crop if requested
# ---------------------------------------------------------
if crop == True:
    bbox = gpd.GeoSeries(
        [Point(bl[1], bl[0]), Point(tr[1], tr[0])],
        crs="EPSG:4326",
    ).to_crs("EPSG:32646")

    geometry = geometry.intersection(bbox.unary_union)

# ---------------------------------------------------------
# Filter InSAR points
# ---------------------------------------------------------

with open(insar_out, "w") as out:

    with open(insar_data) as infile:

        for line in infile:

            if line.startswith("#"):
                out.write(line)
                continue

            lon, lat, displacement, sx, sy, sz = line.split()

            point = gpd.GeoSeries(
                [Point(float(lon), float(lat))],
                crs="EPSG:4326",
            ).to_crs("EPSG:32646").iloc[0]

            if not geometry.contains(point):
                out.write(line)

NameError: name 'geometry' is not defined